# 10. wash_tower 이미지 수집 (Naver Shopping API → staging)

**목적**: `config.NAVER_SEARCH_QUERIES['wash_tower']` 쿼리로 워시타워 이미지 수집

**배경**: Phase 2 준비 — 세탁기+건조기 상하 결합 타워형 제품만 수집.

**흐름**: Naver API → `data/staging/wash_tower/{query}/` → (검수) → `data/processed/wash_tower/`

**수집 제외 대상**
- 세탁기 + 건조기가 **옆으로 나란히** 있는 세트 이미지
- 일반 세탁기 또는 건조기 단독 이미지
- 설치 기사/광고 배경 중심, 텍스트 위주, 부속품 이미지
- 제품 일부만 보이는 이미지

> `data/processed/`와 `data/raw/`는 이 노트북에서 절대 수정하지 않습니다.

In [1]:
import os, sys, re, requests
from io import BytesIO
from pathlib import Path
from PIL import Image as PILImage
from datetime import datetime
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
import config

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

NAVER_ID     = os.getenv('NAVER_CLIENT_ID', '')
NAVER_SECRET = os.getenv('NAVER_CLIENT_SECRET', '')
if not NAVER_ID or not NAVER_SECRET:
    raise EnvironmentError('NAVER_CLIENT_ID / NAVER_CLIENT_SECRET 없음. .env 확인하세요.')
print('Naver API 자격증명 확인 OK')

CLASS_NAME       = 'wash_tower'
QUERIES          = config.NAVER_SEARCH_QUERIES[CLASS_NAME]
IMAGES_PER_QUERY = 100     # 쿼리 수가 적으므로 최대화
MIN_IMG_SIZE     = 150
STAGING_DIR      = os.path.join('data', 'staging', CLASS_NAME)
METADATA_CSV     = os.path.join(config.METADATA_DIR, f'{CLASS_NAME}_staging_metadata.csv')

print(f'검색어 {len(QUERIES)}개  |  쿼리당 최대 {IMAGES_PER_QUERY}장  |  최대 {len(QUERIES)*IMAGES_PER_QUERY}장')
for i, q in enumerate(QUERIES):
    print(f'  {i+1:2d}. {q}')

Naver API 자격증명 확인 OK
검색어 11개  |  쿼리당 최대 100장  |  최대 1100장
   1. 워시타워
   2. LG 워시타워
   3. LG 워시타워 슬림
   4. LG 워시타워 오브제
   5. 삼성 비스포크 그랑데 AI
   6. 삼성 그랑데 AI 세탁건조기
   7. 세탁건조타워
   8. 타워형 세탁건조기
   9. 일체형 세탁건조기 타워
  10. 세탁기 건조기 세트 타워형
  11. 상하 결합 세탁기 건조기


In [2]:
def fetch_items(query, display):
    r = requests.get(
        'https://openapi.naver.com/v1/search/shop.json',
        headers={'X-Naver-Client-Id': NAVER_ID, 'X-Naver-Client-Secret': NAVER_SECRET},
        params={'query': query, 'display': display, 'start': 1},
        timeout=config.DOWNLOAD_TIMEOUT,
    )
    r.raise_for_status()
    return r.json().get('items', [])

def download_and_validate(url, save_path, min_px):
    try:
        r = requests.get(url, timeout=config.DOWNLOAD_TIMEOUT)
        if r.status_code != 200 or 'image' not in r.headers.get('Content-Type', ''):
            return False, 0, 0
        img = PILImage.open(BytesIO(r.content)).convert('RGB')
        w, h = img.size
        if w < min_px or h < min_px:
            return False, w, h
        img.save(save_path, 'JPEG', quality=95)
        return True, w, h
    except Exception:
        return False, 0, 0

records = []
n_saved = 0
n_skip  = 0
n_exist = 0

os.makedirs(config.METADATA_DIR, exist_ok=True)

for q_i, query in enumerate(QUERIES, 1):
    slug  = re.sub(r'[^\w가-힣]', '_', query)
    qdir  = os.path.join(STAGING_DIR, slug)
    os.makedirs(qdir, exist_ok=True)

    try:
        items = fetch_items(query, IMAGES_PER_QUERY)
    except Exception as e:
        print(f'  [{q_i:2d}/{len(QUERIES)}] {query}: API 오류 -> {e}')
        continue

    saved_q = exist_q = skip_q = 0

    for idx, item in enumerate(items):
        img_url = item.get('image', '')
        if not img_url:
            skip_q += 1
            continue
        fname       = f'stg_{idx:04d}.jpg'
        save_path   = os.path.join(qdir, fname)
        title_clean = re.sub(r'<[^>]+>', '', item.get('title', ''))

        if os.path.exists(save_path):
            try:
                img = PILImage.open(save_path)
                w, h = img.size
            except Exception:
                w, h = 0, 0
            records.append({'query': query, 'title': title_clean[:120],
                'link': item.get('link', ''), 'image_url': img_url,
                'saved_path': save_path, 'width': w, 'height': h,
                'status': 'staged',
                'collected_at': datetime.now().isoformat(timespec='seconds')})
            exist_q += 1; n_exist += 1
            continue

        ok, w, h = download_and_validate(img_url, save_path, MIN_IMG_SIZE)
        if ok:
            records.append({'query': query, 'title': title_clean[:120],
                'link': item.get('link', ''), 'image_url': img_url,
                'saved_path': save_path, 'width': w, 'height': h,
                'status': 'staged',
                'collected_at': datetime.now().isoformat(timespec='seconds')})
            saved_q += 1; n_saved += 1
        else:
            skip_q += 1; n_skip += 1

    print(f'  [{q_i:2d}/{len(QUERIES)}] {query:28s} -> 신규 {saved_q:3d}장  기존 {exist_q:3d}장  스킵 {skip_q:3d}장')

print(f'\n수집 완료 -- 신규 {n_saved}장  기존 {n_exist}장  스킵(소형/오류) {n_skip}장')

  [ 1/11] 워시타워                         -> 신규 100장  기존   0장  스킵   0장


  [ 2/11] LG 워시타워                      -> 신규 100장  기존   0장  스킵   0장


  [ 3/11] LG 워시타워 슬림                   -> 신규 100장  기존   0장  스킵   0장


  [ 4/11] LG 워시타워 오브제                  -> 신규 100장  기존   0장  스킵   0장


  [ 5/11] 삼성 비스포크 그랑데 AI               -> 신규 100장  기존   0장  스킵   0장


...

  [ 7/11] 세탁건조타워                       -> 신규 100장  기존   0장  스킵   0장


  [ 8/11] 타워형 세탁건조기                    -> 신규 100장  기존   0장  스킵   0장


  [ 9/11] 일체형 세탁건조기 타워                 -> 신규 100장  기존   0장  스킵   0장


  [10/11] 세탁기 건조기 세트 타워형               -> 신규 100장  기존   0장  스킵   0장


  [11/11] 상하 결합 세탁기 건조기                -> 신규 100장  기존   0장  스킵   0장

수집 완료 -- 신규 1100장  기존 0장  스킵(소형/오류) 0장


In [3]:
new_df = pd.DataFrame(records)
if os.path.exists(METADATA_CSV):
    old_df = pd.read_csv(METADATA_CSV, encoding='utf-8-sig')
    combined = pd.concat([old_df, new_df], ignore_index=True)
    combined.drop_duplicates(subset=['image_url'], keep='last', inplace=True)
else:
    combined = new_df

combined.to_csv(METADATA_CSV, index=False, encoding='utf-8-sig')
print(f'메타데이터 저장: {METADATA_CSV}  ({len(combined)}건)')

print()
print('=== staging 디렉토리 현황 ===')
total_staged = 0
for slug in sorted(os.listdir(STAGING_DIR)):
    d = os.path.join(STAGING_DIR, slug)
    if not os.path.isdir(d):
        continue
    imgs = [f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f'  {slug:40s}: {len(imgs):3d}장')
    total_staged += len(imgs)
print(f'  {"합계":40s}: {total_staged:3d}장')

메타데이터 저장: data\metadata\wash_tower_staging_metadata.csv  (1100건)

=== staging 디렉토리 현황 ===
  LG_워시타워                                 : 100장
  LG_워시타워_슬림                              : 100장
  LG_워시타워_오브제                             : 100장
  삼성_그랑데_AI_세탁건조기                         : 100장
  삼성_비스포크_그랑데_AI                          : 100장
  상하_결합_세탁기_건조기                           : 100장
  세탁건조타워                                  : 100장
  세탁기_건조기_세트_타워형                          : 100장
  워시타워                                    : 100장
  일체형_세탁건조기_타워                            : 100장
  타워형_세탁건조기                               : 100장
  합계                                      : 1100장


In [4]:
import base64

HTML_OUT = os.path.join(config.METADATA_DIR, f'{CLASS_NAME}_staging_review.html')

PRIORITY = {
    '워시타워':              '★★★',
    'LG_워시타워':           '★★★',
    'LG_워시타워_슬림':      '★★★',
    'LG_워시타워_오브제':    '★★★',
    '삼성_비스포크_그랑데_AI': '★★★',
    '삼성_그랑데_AI_세탁건조기': '★★★',
    '세탁건조타워':           '★★☆',
    '타워형_세탁건조기':      '★★☆',
    '일체형_세탁건조기_타워': '★★☆',
    '세탁기_건조기_세트_타워형': '★☆☆',
    '상하_결합_세탁기_건조기': '★☆☆',
}

def img_b64(fpath, px=260):
    try:
        img = PILImage.open(str(fpath)).convert('RGB')
        img.thumbnail((px, px), PILImage.LANCZOS)
        buf = BytesIO()
        img.save(buf, 'JPEG', quality=82)
        return base64.b64encode(buf.getvalue()).decode()
    except Exception:
        return ''

staging_root = Path(STAGING_DIR)
by_q = {}
for d in sorted(staging_root.iterdir()):
    if d.is_dir():
        fs = sorted(d.glob('*.jpg'))
        if fs:
            by_q[d.name] = list(fs)

total_img = sum(len(v) for v in by_q.values())
print(f'HTML 생성 중... 총 {total_img}장')

CSS = ('<style>'
       '*{box-sizing:border-box;margin:0;padding:0}'
       'body{font-family:sans-serif;background:#0f0f0f;color:#ccc;padding:16px}'
       'h1{color:#fff;margin-bottom:4px;font-size:22px}'
       '.sub{color:#888;font-size:12px;margin-bottom:14px}'
       '.warn{background:#2a1010;border:2px solid #c0392b;border-radius:6px;'
       'padding:12px 16px;margin-bottom:18px;font-size:13px;line-height:1.9}'
       '.warn b{color:#e74c3c}'
       '.qh{background:#1a2a3a;color:#7ec8e3;font-size:13px;font-weight:bold;'
       'padding:7px 12px;border-radius:6px 6px 0 0;'
       'border-left:4px solid #7ec8e3;margin-top:18px}'
       '.grid{display:flex;flex-wrap:wrap;gap:5px;padding:8px;'
       'background:#161616;border-radius:0 0 6px 6px}'
       '.card{width:138px;background:#1e1e1e;border-radius:3px;'
       'overflow:hidden;border:1px solid #2a2a2a}'
       '.card:hover{border-color:#7ec8e3}'
       '.card img{width:138px;height:118px;object-fit:contain;'
       'background:#0a0a0a;display:block}'
       '.card .m{padding:4px;font-size:8.5px;color:#666;word-break:break-all;line-height:1.4}'
       '</style>')

WARN = ('<div class="warn">'
        '<b>⚠ 제거 대상 — staging 폴더에서 직접 파일 삭제</b><br>'
        '① 세탁기+건조기가 <b>옆으로 나란히</b> 있는 세트 이미지<br>'
        '② <b>일반 세탁기 또는 건조기 단독</b> 이미지<br>'
        '③ 설치 기사·광고·텍스트 위주 / 상세페이지 이미지<br>'
        '④ 부속품·받침대·설치 키트<br>'
        '⑤ 제품 일부만 보이는 이미지<br>'
        '<br><b style="color:#f39c12">★☆☆ 쿼리</b>(세탁기 건조기 세트 타워형 / 상하 결합)는 '
        '옆으로 나란히 세트가 많음 — 특히 꼼꼼히 검수하세요.'
        '</div>')

parts = []
for slug, files in by_q.items():
    pri  = PRIORITY.get(slug, '★☆☆')
    qn   = slug.replace('_', ' ')
    cards = []
    for fp in files:
        b   = img_b64(fp)
        rel = fp.parent.name + '/' + fp.name
        img_tag = f'<img src="data:image/jpeg;base64,{b}" loading="lazy">' if b else ''
        cards.append(f'<div class="card">{img_tag}'
                     f'<div class="m">{fp.name}<br>'
                     f'<span style="color:#444">{rel}</span></div></div>')
    parts.append(f'<div class="qh">{pri} {qn} '
                 f'<span style="color:#888;font-size:11px">({len(files)}장)</span></div>'
                 f'<div class="grid">{"".join(cards)}</div>')

html = ('<!DOCTYPE html><html><head><meta charset="utf-8">'
        '<title>wash_tower staging</title>'
        + CSS + '</head><body>'
        '<h1>wash_tower Staging 검수</h1>'
        f'<div class="sub">총 {total_img}장 | Naver Shopping API | processed 이동 전 검수 단계</div>'
        + WARN + ''.join(parts) + '</body></html>')

with open(HTML_OUT, 'w', encoding='utf-8') as fh:
    fh.write(html)
print(f'[OK] {HTML_OUT}  ({os.path.getsize(HTML_OUT) // 1024} KB)')

HTML 생성 중... 총 1100장


[OK] data\metadata\wash_tower_staging_review.html  (11051 KB)


In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

staged_files = []
for slug in sorted(os.listdir(STAGING_DIR)):
    qd = os.path.join(STAGING_DIR, slug)
    if not os.path.isdir(qd):
        continue
    for f in sorted(os.listdir(qd)):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            staged_files.append((slug, os.path.join(qd, f)))

by_query = {}
for slug, fpath in staged_files:
    by_query.setdefault(slug, []).append(fpath)

SAMPLE   = 5
n_q      = len(by_query)
fig, axes = plt.subplots(n_q, SAMPLE, figsize=(SAMPLE * 3, n_q * 3))
if n_q == 1:
    axes = [axes]
axes = np.array(axes).reshape(n_q, SAMPLE)

for r_i, (slug, paths) in enumerate(sorted(by_query.items())):
    for c_i in range(SAMPLE):
        ax = axes[r_i, c_i]
        if c_i < len(paths):
            try:
                ax.imshow(PILImage.open(paths[c_i]).convert('RGB'))
                ax.set_title(f'{slug[:20]}\n{os.path.basename(paths[c_i])}', fontsize=6)
            except Exception:
                ax.text(0.5, 0.5, 'ERR', ha='center', va='center',
                        transform=ax.transAxes)
        ax.axis('off')

plt.suptitle(f'wash_tower staging — 쿼리별 샘플 {SAMPLE}장', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('다음 단계: 08_approve_staging.ipynb의 CLASS_NAME을 wash_tower로 변경 후 검수')

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 50892 (\N{HANGUL SYLLABLE WEO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 49884 (\N{HANGUL SYLLABLE SI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 53440 (\N{HANGUL SYLLABLE TA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 49836 (\N{HANGUL SYLLABLE SEUL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 47548 (\N{HANGUL SYLLABLE RIM}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 50724 (\N{HANGUL SYLLABLE O}) missing from f

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 48708 (\N{HANGUL SYLLABLE BI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 49828 (\N{HANGUL SYLLABLE SEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 54252 (\N{HANGUL SYLLABLE PO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 53356 (\N{HANGUL SYLLABLE KEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 49345 (\N{HANGUL SYLLABLE SANG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 54616 (\N{HANGUL SYLLABLE HA}) missing from 

다음 단계: 08_approve_staging.ipynb의 CLASS_NAME을 wash_tower로 변경 후 검수


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 51068 (\N{HANGUL SYLLABLE IL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 52404 (\N{HANGUL SYLLABLE CE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 53244 (\N{HANGUL SYLLABLE KWEO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 47532 (\N{HANGUL SYLLABLE RI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 48324 (\N{HANGUL SYLLABLE BYEOL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_11024\1931982725.py:39: UserWarning: Glyph 49368 (\N{HANGUL SYLLABLE SAEM}) missing fr